In [1]:
!pip install --upgrade binance-historical-data plotly -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.8/272.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.3 MB/s eta 0:00:00


In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from tqdm.notebook import tqdm
from plotly.subplots import make_subplots
from datetime import datetime, timedelta, date, time
from binance_historical_data import BinanceDataDumper
warnings.filterwarnings('ignore')

In [3]:
def get_list_all_trading_pairs():
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_list_all_trading_pairs()

def filter_usdt_tickers(tickers):
    exclude_keywords = ["UPUSDT", "DOWNUSDT", "BEARUSDT", "BULLUSDT"]
    return [ticker for ticker in tickers if ticker.endswith("USDT") and not any(ex in ticker for ex in exclude_keywords)]

def find_first_data_date(ticker):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_min_start_date_for_ticker(ticker)

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def detect_timestamp_unit(timestamp):
    num_digits = len(str(timestamp))
    if num_digits == 13:
        return 'ms'
    elif num_digits == 16:
        return 'us'
    else:
        raise ValueError(f"Timestamp không hợp lệ: {timestamp}")

def convert_timestamp(timestamp):
    unit = detect_timestamp_unit(timestamp)
    return pd.to_datetime(timestamp, unit=unit, errors='coerce')

def download_ticker(ticker, date_start, date_end, data_frequency="1h"):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency= data_frequency,
    )
    date_start = datetime.strptime(date_start, "%Y-%m-%d").date()
    date_end = datetime.strptime(date_end, "%Y-%m-%d").date()
    data_dumper.dump_data(
        tickers = ticker,
        date_start = date_start,
        date_end = date_end,
        is_to_update_existing = False,
    )

def read_csv_file(file_path):
    df = pd.read_csv(file_path)
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    df['open_time'] = df['open_time'].apply(convert_timestamp)
    df['close_time'] = df['close_time'].apply(convert_timestamp)
    return df

def get_csv_files(directory):
    try:
        if not os.path.exists(directory):
            print(f"Warning: Thư mục không tồn tại: {directory}")
            return []
        return [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.csv')]
    except Exception as e:
        print(f"Lỗi khi đọc thư mục {directory}: {str(e)}")
        return []

# Load data from file
def process_csv_files(ticker, data_frequency = "1h"):
    daily_path = os.path.join(os.getcwd(), f"spot/daily/klines/{ticker}/{data_frequency}")
    monthly_path = os.path.join(os.getcwd(), f"spot/monthly/klines/{ticker}/{data_frequency}")
    daily_files = get_csv_files(daily_path)
    monthly_files = get_csv_files(monthly_path)
    all_files = daily_files + monthly_files
    if not all_files:
        print(f"❗ Không có file CSV nào cho {ticker}")
        return None
    data = pd.concat([read_csv_file(file) for file in all_files], ignore_index=True)
    data.sort_values(by='open_time', inplace=True)
    return data

In [4]:
ticker = 'BTCUSDT'
data_frequency = '15m'
download_ticker(ticker, "2025-06-01", "2025-07-17", data_frequency)
data = process_csv_files(ticker, data_frequency)
data = data.drop(columns=['close_time',	'quote_asset_volume',	'number_of_trades',	'taker_buy_base_asset_volume',	'taker_buy_quote_asset_volume',	'ignore'])
data


---> Found overall tickers: 587
---> Filter to asked tickers: 7
------> Tickers left: 2
Download full data for 2 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 15m
---> Start Date: 20250601
---> End Date: 20250716


Tickers:   0%|          | 0/2 [00:00<?, ?it/s]

monthly files to download:   0%|          | 0/1 [00:00<?, ?files/s]

daily files to download:   0%|          | 0/16 [00:00<?, ?files/s]

monthly files to download: 0files [00:00, ?files/s]

daily files to download:   0%|          | 0/16 [00:00<?, ?files/s]

Tried to dump data for 2 tickers:
---> For BTCUSDT new data saved for: 1 months 15 days
---> For BTCUSD new data saved for: 0 months 0 days


,open_time,Open,High,Low,Close,volume
1425,2025-06-01 00:15:00,104473.01,104490.57,104370.26,104380.96,95.60659
1426,2025-06-01 00:30:00,104380.95,104455.96,104320.02,104455.34,70.32377
1427,2025-06-01 00:45:00,104455.34,104480.00,104390.24,104446.49,36.63954
1428,2025-06-01 01:00:00,104446.49,104470.65,104349.05,104470.65,62.15181
1429,2025-06-01 01:15:00,104470.65,104505.58,104311.12,104382.54,86.09050
...,...,...,...,...,...,...
755,2025-07-15 22:45:00,117729.27,117868.52,117655.10,117827.74,119.91992
756,2025-07-15 23:00:00,117827.74,117972.86,117741.93,117833.00,168.19499
757,2025-07-15 23:15:00,117833.01,117859.61,117556.52,117576.54,75.58643
758,2025-07-15 23:30:00,117576.53,117784.65,117500.00,117784.65,87.72558


In [5]:
def get_periods_per_year(freq: str) -> int:
    if freq.endswith('m'):  # phút
        minutes = int(freq[:-1])
        return int(365 * 24 * 60 / minutes)
    elif freq.endswith('h'):  # giờ
        hours = int(freq[:-1])
        return int(365 * 24 / hours)
    elif freq.endswith('d'):  # ngày
        return 365
    elif freq.endswith('w'):  # tuần
        return 52
    else:
        raise ValueError(f"Không hỗ trợ định dạng data_frequency: {freq}")

def calculate_sharpe_ratio(returns_series: pd.Series, data_frequency: str, annualized_risk_free_rate: float = 0.03) -> float:
    """
    Tính toán Sharpe Ratio hàng năm từ một chuỗi lợi nhuận theo kỳ.

    Args:
        returns_series (pd.Series): Chuỗi lợi nhuận của chiến lược theo từng kỳ.
        data_frequency (str): Tần suất của dữ liệu ('15m', '1H', '1D').
        annualized_risk_free_rate (float): Lãi suất phi rủi ro hàng năm.

    Returns:
        float: Giá trị Sharpe Ratio đã được quy đổi ra năm.
    """
    # Xử lý trường hợp không có giao dịch hoặc không có biến động
    if returns_series.std() == 0 or pd.isna(returns_series.std()):
        return 0.0

    # 1. Lấy số kỳ trong một năm
    periods_per_year = get_periods_per_year(data_frequency)

    # 2. Quy đổi lãi suất phi rủi ro ra theo từng kỳ
    rf_per_period = annualized_risk_free_rate / periods_per_year

    # 3. Tính toán Sharpe Ratio
    # Lợi nhuận trung bình mỗi kỳ trừ đi lãi suất phi rủi ro mỗi kỳ
    excess_return_mean = returns_series.mean() - rf_per_period

    # Sharpe Ratio hàng ngày = Lợi nhuận vượt trội trung bình / Độ lệch chuẩn lợi nhuận
    sharpe_ratio = excess_return_mean / returns_series.std()

    # 4. Quy đổi ra Sharpe Ratio hàng năm
    annualized_sharpe = sharpe_ratio * np.sqrt(periods_per_year)

    return annualized_sharpe

def backtest_vwap(df, a, b, fee_rate=0.00075):
    """
    Thực hiện backtest cho chiến lược giao cắt Rolling VWAP (MVWAP).
    Đây là chiến lược Long/Short, có tính đến phí giao dịch.

    Args:
        df (pd.DataFrame): DataFrame dữ liệu.
        a (int): Cửa sổ của MVWAP nhanh.
        b (int): Cửa sổ của MVWAP chậm.
        fee_rate (float): Tỷ lệ phí giao dịch (ví dụ: 0.075% = 0.00075).

    Returns:
        pd.Series: Chuỗi lợi nhuận của chiến lược sau khi đã trừ phí.
    """
    # 1. Tính toán hai đường MVWAP
    mvwap_fast = df['tp_vol'].rolling(window=a).sum() / (df['volume'].rolling(window=a).sum() + 1e-9)
    mvwap_slow = df['tp_vol'].rolling(window=b).sum() / (df['volume'].rolling(window=b).sum() + 1e-9)

    # 2. Tạo tín hiệu Long/Short (-1 = Short, 1 = Long)
    signal = np.sign(mvwap_fast - mvwap_slow)

    # 3. Tính toán lợi nhuận thị trường và dịch chuyển tín hiệu
    market_returns = df['Close'].pct_change()
    shifted_signal = signal.shift(1).fillna(0)

    # 4. Tính lợi nhuận chiến lược (chưa có phí)
    strategy_returns = market_returns * shifted_signal

    # 5. TÍNH TOÁN PHÍ GIAO DỊCH
    # Một giao dịch xảy ra khi tín hiệu thay đổi
    # .diff() sẽ khác 0 khi signal thay đổi (ví dụ: từ 1 sang -1, .diff() = -2)
    trades = shifted_signal.diff().ne(0)
    trade_count = trades.sum()
    trade_times = df.index[trades].tolist()
    print (trade_count)
    print (trade_times)

    # Trừ phí vào lợi nhuận tại những cây nến có giao dịch
    strategy_returns[trades] -= fee_rate

    return strategy_returns.fillna(0)

def sweep_vwap(df, a_range, b_range):
    """
    Quét qua các tham số khác nhau của chiến lược MVWAP crossover.

    Args:
        df (pd.DataFrame): DataFrame dữ liệu.
        a_range (range): Dải tham số cho MVWAP nhanh.
        b_range (range): Dải tham số cho MVWAP chậm.

    Returns:
        dict: Một dictionary với key là (a, b) và value là Sharpe Ratio.
    """
    metrics = {}

    # Sử dụng tqdm để tạo thanh tiến trình đẹp mắt
    for a in tqdm(a_range, desc="Quét MVWAP"):
        for b in b_range:
            # Đảm bảo a < b để logic giao cắt nhanh/chậm có ý nghĩa
            if a < b:
                # Chạy backtest cho cặp tham số (a,b)
                bt_returns = backtest_vwap(df, a, b)

                # Tính Sharpe Ratio (một thước đo hiệu suất chuẩn)
                metrics[(a, b)] = calculate_sharpe_ratio(bt_returns, data_frequency)

    return metrics

In [6]:
# Tính giá điển hình (Typical Price) - hlc3
# Đây là "giá đại diện" cho mỗi cây nến, đúng như trong nghiên cứu
data['hlc3'] = (data['High'] + data['Low'] + data['Close']) / 3
# Tính giá trị giao dịch (Typical Price x Volume)
# Chuẩn bị sẵn tử số cho công thức VWAP
data['tp_vol'] = data['hlc3'] * data['volume']

# ======================================================================
# THỰC THI SWEEP TEST
# ======================================================================

# Định nghĩa dải tham số để quét

a_range = range(2, 100, 2)  # Dải chi tiết hơn
b_range = range(2, 610, 2)

# Chạy sweep test
met = sweep_vwap(data, a_range, b_range)

# Sắp xếp kết quả để xem 5 cặp tham số tốt nhất
sorted_met = sorted(met.items(), key=lambda item: item[1], reverse=True)

print("\n--- 5 cặp tham số MVWAP tốt nhất (dựa trên Sharpe Ratio) ---")
for params, sharpe in sorted_met[:5]:
    print(f"MVWAP Nhanh: {params[0]}, MVWAP Chậm: {params[1]} -> Sharpe Ratio (năm): {sharpe:.2f}")

Quét MVWAP:   0%|          | 0/49 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
[1425, 1703, 1794, 1992, 2502, 2812, 3027, 3281, 3322, 3607, 3976, 4083, 4291, 1204, 825, 345, 944, 1133, 687]
19
[1425, 1705, 1794, 1992, 2502, 2814, 3027, 3281, 3322, 3607, 3977, 4084, 4291, 1204, 825, 346, 944, 1133, 687]
19
[1425, 1707, 1793, 1993, 2503, 2815, 3027, 3282, 3322, 3607, 3977, 4085, 4292, 1204, 825, 346, 944, 1133, 687]
19
[1425, 1709, 1793, 1993, 2504, 2816, 3027, 3282, 3321, 3608, 3978, 4085, 4292, 1204, 825, 348, 944, 1133, 687]
19
[1425, 1711, 1794, 1993, 2505, 2817, 3027, 3282, 3321, 3608, 3978, 4085, 4292, 1204, 825, 348, 945, 1133, 687]
19
[1425, 1713, 1794, 1994, 2505, 2819, 3028, 3282, 3322, 3609, 3979, 4086, 4292, 1204, 826, 349, 945, 1133, 687]
19
[1425, 1715, 1794, 1994, 2506, 2819, 3028, 3282, 3322, 3609, 3980, 4085, 4292, 1204, 826, 350, 945, 1133, 687]
21
[1425, 1717, 1794, 1994, 2507, 2820, 3028, 3282, 3323, 3609, 3981, 4082, 4083, 4086, 4293, 1204, 827, 352, 946, 1133, 688]
19
[1425, 1719, 1794, 1994, 

In [7]:
# ======================================================================
# BƯỚC 0: AN TOÀN DỮ LIỆU (Giữ nguyên)
# ======================================================================
data['open_time'] = pd.to_datetime(data['open_time'])
data = data.sort_values(by='open_time')

# ======================================================================
# BƯỚC 1: LẤY TOP 5 CHIẾN LƯỢC VÀ CHUẨN BỊ DỮ LIỆU
# ======================================================================
sorted_met = sorted(met.items(), key=lambda item: item[1], reverse=True)
top_params = [item[0] for item in sorted_met[:10]]
top_params.append((34,89))

results_to_plot = {}
for params in top_params:
    a, b = params
    returns = backtest_vwap(data, a, b)
    equity_curve = (1 + returns).cumprod()
    results_to_plot[params] = {'equity': equity_curve}

# ======================================================================
# BƯỚC 2: TÍNH TOÁN HIỆU SUẤT CỦA CHIẾN LƯỢC "BUY & HOLD"
# ======================================================================
# 2.1 Tính lợi nhuận theo từng kỳ của việc chỉ giữ BTC
buy_and_hold_returns = data['Close'].pct_change().fillna(0)

# 2.2 Tính đường cong vốn cho Buy & Hold
buy_and_hold_equity = (1 + buy_and_hold_returns).cumprod()

# ======================================================================
# BƯỚC 3: VẼ BIỂU ĐỒ SO SÁNH HIỆU SUẤT DUY NHẤT
# ======================================================================
# Khởi tạo một biểu đồ đơn
fig = go.Figure()

# 3.1 Vẽ các đường cong vốn của Top 5 chiến lược
for params, result_data in results_to_plot.items():
    fig.add_trace(
        go.Scatter(
            x=data['open_time'],
            y=result_data['equity'],
            mode='lines',
            name=f'Equity ({params[0]}, {params[1]})'
        )
    )

# 3.2 Vẽ đường cong vốn của chiến lược Buy & Hold để làm đường cơ sở
fig.add_trace(
    go.Scatter(
        x=data['open_time'],
        y=buy_and_hold_equity,
        mode='lines',
        name=f'Buy & Hold {ticker}',
        line=dict(color='black', width=2, dash='dash') # Làm cho nó nổi bật
    )
)

# 3.3 Cấu hình giao diện biểu đồ
fig.update_layout(
    title_text='So sánh Hiệu suất: Top 5 Chiến lược Giao cắt MVWAP vs. Buy & Hold',
    height=700,
    xaxis_title='Thời gian',
    yaxis_title='Vốn (chuẩn hóa)',
    legend_title_text='Chiến lược',
    hovermode='x unified',
    yaxis_type="log" # Sử dụng trục y log để so sánh tăng trưởng phần trăm tốt hơn
)

fig.show()

31
[1425, 1651, 1767, 1952, 2461, 2766, 2815, 2822, 2852, 2868, 2999, 3199, 3206, 3249, 3261, 3278, 3314, 3558, 3922, 3939, 3952, 4039, 4243, 1173, 796, 338, 907, 1088, 1253, 1261, 454]
23
[1425, 1651, 1769, 1958, 2469, 2770, 2854, 2877, 3003, 3278, 3317, 3563, 3932, 3950, 3952, 4046, 4251, 1177, 803, 339, 910, 1091, 465]
35
[1425, 1655, 1767, 1787, 1788, 1953, 2462, 2767, 2817, 2821, 2852, 2869, 2999, 3203, 3204, 3249, 3263, 3278, 3314, 3559, 3923, 3939, 3952, 4039, 4243, 1174, 797, 806, 811, 338, 907, 1088, 1253, 1262, 454]
23
[1425, 1653, 1769, 1958, 2469, 2770, 2854, 2877, 3003, 3278, 3317, 3563, 3932, 3950, 3952, 4046, 4251, 1178, 804, 339, 910, 1091, 465]
31
[1425, 1673, 1767, 1787, 1788, 1955, 2465, 2768, 2853, 2867, 3000, 3253, 3271, 3278, 3313, 3562, 3952, 4039, 4245, 1174, 814, 339, 908, 1088, 1253, 1262, 455, 458, 462, 471, 666]
35
[1425, 1653, 1767, 1787, 1788, 1952, 2462, 2766, 2816, 2821, 2852, 2869, 2999, 3201, 3206, 3249, 3263, 3278, 3314, 3559, 3923, 3939, 3952, 4039, 